# LG Gram Low-Rating Review Crawler & Negative Experience Analysis (LG Official Site)

목표: LG 공식 사이트 그램 제품 페이지에서 **별점 낮은 리뷰부터** 수집 → 전처리 → 문제점(Cons) 카테고리화 → 감정/부정 경험 시각화.

제품 기준 URL: https://www.lge.co.kr/care-solutions/notebook/16z90tp-ka5wk?dpType=careTab

버전: v0.1 (Selector draft & pipeline scaffold) | 작성일: 2025-08-18

> 실행 순서 권장: 1 → 2 → 3 ... (대규모 수집 전 7, 8, 9 섹션으로 셀렉터 검증 필수)

---


## 1. Install & Verify Dependencies
(필요시 1회 실행) 이미 설치되어 있으면 건너뜀.
- selenium, webdriver-manager
- pandas, numpy, tqdm, beautifulsoup4
- nltk, konlpy (Optional, Fallback 제공)
- wordcloud, seaborn, matplotlib, networkx


In [1]:
# Silent install / verify
import sys, subprocess, importlib, pkgutil
packages = [
    'selenium','webdriver-manager','pandas','numpy','tqdm','beautifulsoup4',
    'nltk','wordcloud','seaborn','matplotlib','networkx'
]
optional = ['konlpy']
for p in packages + optional:
    try:
        importlib.import_module(p)
    except ImportError:
        print(f'Installing {p} ...')
        subprocess.check_call([sys.executable,'-m','pip','install',p])
print('✅ Dependencies ready')

Installing webdriver-manager ...
Installing beautifulsoup4 ...
Installing beautifulsoup4 ...
✅ Dependencies ready
✅ Dependencies ready


## 2. Configuration & Paths
BASE_URL / 대체 모델 URL / 데이터 디렉토리 및 파일 경로 설정, 랜덤 슬립/시드 고정.

In [2]:
from pathlib import Path
import os, random, time, datetime as dt
import json, re, math, hashlib, textwrap
import numpy as np
import pandas as pd

BASE_URL = 'https://www.lge.co.kr/care-solutions/notebook/16z90tp-ka5wk?dpType=careTab'
ALT_MODEL_URLS = [
    'https://www.lge.co.kr/care-solutions/notebook/17z90ts-ka58k',
    'https://www.lge.co.kr/care-solutions/notebook/16z90tp-ka5wk',
    'https://www.lge.co.kr/care-solutions/notebook/14z90ts-ka58k',
]

DATA_DIR = Path('03.CX_Group4/02.LG_Gram/data/lg_official_low')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_JSONL = DATA_DIR / 'reviews_low.jsonl'
PROC_CSV = DATA_DIR / 'reviews_processed.csv'
NEG_CSV = DATA_DIR / 'reviews_negative.csv'
FIG_DIR = DATA_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
LEX_SNAPSHOT = DATA_DIR / 'negative_lexicon.json'

random.seed(42)
np.random.seed(42)

def rand_sleep(a=0.6,b=1.8):
    time.sleep(random.uniform(a,b))

print('Paths ready:', DATA_DIR)

Paths ready: 03.CX_Group4\02.LG_Gram\data\lg_official_low


## 3. Imports, NLP Tokenizer Fallback, Font Setup

In [3]:
import nltk
nltk.download('punkt', quiet=True)
try:
    from konlpy.tag import Okt
    _OKT_AVAILABLE=True
    okt = Okt()
except Exception as e:
    print('[WARN] Okt unavailable -> fallback regex tokenizer:', e)
    _OKT_AVAILABLE=False
    class FallbackTokenizer:
        def morphs(self, text):
            return re.findall(r'[가-힣]{2,}|[A-Za-z]{2,}', text)
    okt = FallbackTokenizer()

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import networkx as nx

# Windows Malgun Gothic
mpl.rc('font', family='Malgun Gothic')
mpl.rcParams['axes.unicode_minus']=False
print('Okt available:', _OKT_AVAILABLE)

[WARN] Okt unavailable -> fallback regex tokenizer: No JVM shared library file (jvm.dll) found. Try setting up the JAVA_HOME environment variable properly.
Okt available: False


## 4. Negative Category Lexicon & Stopwords
카테고리 정의 + 확장 함수 (동적으로 새로운 키워드 추가 가능).

In [4]:
STOPWORDS = set(['그리고','그러나','하지만','에서','으로','하다','했다','하여','그냥','이번','제품','노트북','그램','사용','사용중','사용후','정도','조금','많이'])
NEGATIVE_KEYWORDS = {
  'performance': ['느리','버벅','렉','지연','멈추','속도','성능','답답'],
  'battery': ['배터리','방전','충전','잔량','배터리수명','전원'],
  'heat': ['발열','뜨거','과열','열나','온도'],
  'noise': ['소음','웅웅','팬소리','팬돌','시끄럽'],
  'keyboard': ['키보드','키감','travel','입력안','오타','백라이트'],
  'build_quality': ['마감','갈라짐','깨짐','헐거','유격','스크래치','뒤틀'],
  'screen': ['화면','빛샘','밝기','색감','잔상','픽셀','패널'],
  'portability': ['무게','휴대','들고다니','가벼','무겁'],
  'price_value': ['비싸','가격','가성비','값어치','할인없'],
  'software': ['드라이버','업데이트','윈도','에러','오류','프로그램','앱'],
  'after_service': ['AS','서비스','센터','수리','교환','환불','고객센터']
}

def extend_lexicon(category, keywords):
    NEGATIVE_KEYWORDS.setdefault(category, [])
    for kw in keywords:
        if kw not in NEGATIVE_KEYWORDS[category]:
            NEGATIVE_KEYWORDS[category].append(kw)

NEG_FLAT = {kw:cat for cat, kws in NEGATIVE_KEYWORDS.items() for kw in kws}
print('Lexicon categories:', list(NEGATIVE_KEYWORDS.keys()))

Lexicon categories: ['performance', 'battery', 'heat', 'noise', 'keyboard', 'build_quality', 'screen', 'portability', 'price_value', 'software', 'after_service']


## 5. Selenium Driver Factory & Robust Interaction Helpers

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
from webdriver_manager.chrome import ChromeDriverManager

USER_AGENTS = [
 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0'
]

def init_driver():
    ua = random.choice(USER_AGENTS)
    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument(f'--user-agent={ua}')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_experimental_option('useAutomationExtension', False)
    from selenium.webdriver.chrome.service import Service
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    driver.set_window_size(1400, 1000)
    return driver

def with_retry(fn, retries=3, base=1.5):
    for i in range(retries):
        try:
            return fn()
        except Exception as e:
            if i == retries-1:
                raise
            time.sleep(base*(i+1))

def safe_find(driver, by, sel, timeout=6):
    try:
        return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((by, sel)))
    except TimeoutException:
        return None

def safe_click(driver, by, sel, timeout=6):
    el = safe_find(driver, by, sel, timeout)
    if not el:
        return False
    try:
        el.click(); return True
    except ElementClickInterceptedException:
        driver.execute_script('arguments[0].click();', el); return True
    except Exception:
        return False

print('Driver helpers ready')

Driver helpers ready


## 6. CSS Selectors Definition & Version Tag
사이트 DOM 변동 가능 — 수시 점검 필요. (version_ts: 2025-08-18T00:00Z)

In [6]:
SEL = {
 'review_area': '#reviewArea',
 'review_list_container': '.lge-review-wrap',
 'review_item': '.review-item, .review, [class*="review"]',
 'rating': '.rating, .star-rating, [class*="star"]',
 'content': '.review-content, .comment, [class*="content"]',
 'author': '.review-author, .user-name, [class*="author"]',
 'date': '.review-date, .date, [class*="date"]',
 'sort_dropdown': '.sort-select, select',
 'sort_lowest_option': 'option[value*="low"], option[value*="asc"]',
 'load_more': '.btn-review-more, #reviewMoreBtn, .more, button.more',
}
for k,v in SEL.items():
    print(f'{k:20s} -> {v}')

review_area          -> #reviewArea
review_list_container -> .lge-review-wrap
review_item          -> .review-item, .review, [class*="review"]
rating               -> .rating, .star-rating, [class*="star"]
content              -> .review-content, .comment, [class*="content"]
author               -> .review-author, .user-name, [class*="author"]
date                 -> .review-date, .date, [class*="date"]
sort_dropdown        -> .sort-select, select
sort_lowest_option   -> option[value*="low"], option[value*="asc"]
load_more            -> .btn-review-more, #reviewMoreBtn, .more, button.more


## 7. Selector Debug Function (Sample Extraction)

In [7]:
def debug_selectors(sample=3, sort_low=True):
    d = init_driver()
    try:
        target = BASE_URL + ('#review' if '#review' not in BASE_URL else '')
        d.get(target)
        rand_sleep(2,3)
        # 정렬 시도
        if sort_low:
            if safe_click(d, By.CSS_SELECTOR, SEL['sort_dropdown']):
                rand_sleep(0.5,1.0)
                safe_click(d, By.CSS_SELECTOR, SEL['sort_lowest_option'])
        container = safe_find(d, By.CSS_SELECTOR, SEL['review_list_container']) or d
        items = container.find_elements(By.CSS_SELECTOR, SEL['review_item'])
        print('Found review elements:', len(items))
        for it in items[:sample]:
            txt = it.text.split('\n')[:6]
            print(' -', ' | '.join(txt))
        if not items:
            print('[WARN] No review elements, re-check selectors.')
    finally:
        d.quit()

# debug_selectors()  # 수동 실행

## 8. Review Element Parsing Helpers

In [8]:
from bs4 import BeautifulSoup

def load_existing_ids():
    ids=set()
    if RAW_JSONL.exists():
        with open(RAW_JSONL,'r',encoding='utf-8') as f:
            for line in f:
                try:
                    obj=json.loads(line); ids.add(obj.get('review_id'))
                except: pass
    return ids

def parse_review_elem(elem):
    html = elem.get_attribute('innerHTML')
    soup = BeautifulSoup(html, 'html.parser')
    def pick(sel):
        t = soup.select_one(sel)
        return t.get_text(strip=True) if t else ''
    rating_raw = pick(SEL['rating'])
    rating=None
    if rating_raw:
        m=re.search(r'(\d+(?:\.\d)?)', rating_raw)
        if m: rating=float(m.group(1))
    content = pick(SEL['content']) or elem.text
    author = pick(SEL['author'])
    date = pick(SEL['date'])
    rid_src = f'{content[:80]}|{date}|{rating}'
    review_id = hashlib.md5(rid_src.encode('utf-8')).hexdigest()
    return {
        'review_id': review_id,
        'rating': rating,
        'date': date,
        'author': author,
        'content': content,
        'raw_html': html
    }
print('Parsing helpers ready')

Parsing helpers ready


## 9. Incremental Crawl Function (Load More Pagination)

In [9]:
def crawl_reviews(max_reviews=100, max_clicks=30):
    existing = load_existing_ids()
    mode = 'a' if RAW_JSONL.exists() else 'w'
    collected = 0
    d = init_driver()
    try:
        target = BASE_URL + ('#review' if '#review' not in BASE_URL else '')
        d.get(target)
        rand_sleep(2,3)
        # 정렬 낮은 별점 (가능시)
        if safe_click(d, By.CSS_SELECTOR, SEL['sort_dropdown']):
            rand_sleep(0.5,1.0)
            safe_click(d, By.CSS_SELECTOR, SEL['sort_lowest_option'])
            rand_sleep(1,2)
        clicks = 0
        with open(RAW_JSONL, mode, encoding='utf-8') as fw:
            while collected < max_reviews and clicks <= max_clicks:
                container = safe_find(d, By.CSS_SELECTOR, SEL['review_list_container']) or d
                elems = container.find_elements(By.CSS_SELECTOR, SEL['review_item'])
                print(f'Page batch reviews found: {len(elems)}')
                new_in_batch=0
                for e in elems:
                    if collected >= max_reviews: break
                    try:
                        data = parse_review_elem(e)
                        if not data['content'] or len(data['content']) < 5:
                            continue
                        if data['review_id'] in existing:
                            continue
                        fw.write(json.dumps(data, ensure_ascii=False)+'\n')
                        existing.add(data['review_id'])
                        collected += 1
                        new_in_batch += 1
                        print(f'Collected {collected}: {data["content"][:40]}...')
                    except Exception as er:
                        print('Parse error:', er)
                if collected >= max_reviews:
                    break
                # 더보기
                more = safe_find(d, By.CSS_SELECTOR, SEL['load_more'])
                if more:
                    try:
                        d.execute_script('arguments[0].click();', more)
                        clicks += 1
                        print('Clicked load more:', clicks)
                        rand_sleep(2,3)
                        continue
                    except Exception as e:
                        print('Load more click failed:', e)
                        break
                else:
                    print('No more button — stopping.')
                    break
                if new_in_batch == 0:
                    print('No new reviews in batch — stopping.')
                    break
    finally:
        d.quit()
    print(f'✅ Crawl done. Newly collected: {collected}')
    return collected

# small test (manual)
# crawl_reviews(20,5)

## 10. Multi-Model Crawl Loop (Optional)
대체 모델 페이지 순회하면서 신규 리뷰 탐색 (중복 회피).

In [10]:
def multi_model_crawl(urls=None, per_url_limit=40):
    global BASE_URL
    urls = urls or ALT_MODEL_URLS
    baseline_ids = load_existing_ids()
    before = len(baseline_ids)
    for u in urls:
        print('\n[Model]', u)
        BASE_URL = u
        collected = crawl_reviews(max_reviews=per_url_limit, max_clicks=15)
        after_ids = load_existing_ids()
        if len(after_ids) > before:
            print('New reviews discovered; stopping multi crawl early.')
            break
    print('Total unique review ids now:', len(load_existing_ids()))

# multi_model_crawl()

## 11. Raw JSONL -> Processed DataFrame Builder

In [11]:
def parse_date_flex(s):
    if not isinstance(s,str): return None
    for fmt in ('%Y.%m.%d','%Y-%m-%d','%y.%m.%d','%Y/%m/%d'):
        try: return dt.datetime.strptime(s, fmt)
        except: pass
    return None

def build_processed():
    if not RAW_JSONL.exists():
        print('No raw file; crawl first.')
        return None
    rows=[]
    with open(RAW_JSONL,'r',encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: pass
    if not rows:
        print('Empty raw.')
        return None
    df = pd.DataFrame(rows)
    if 'content' not in df:
        print('content column missing')
        return None
    # defers text ops to later functions
    df['date_parsed'] = df.get('date','').map(parse_date_flex)
    df.to_csv(PROC_CSV, index=False, encoding='utf-8-sig')
    print('Built processed skeleton:', len(df))
    return df

# build_processed()

## 12. Text Cleaning, Tokenization, Category & Sentiment Functions

In [12]:
def clean_text(t):
    if not isinstance(t,str): return ''
    t = re.sub(r'\s+',' ', t)
    t = re.sub(r'["\'`]+','', t)
    return t.strip()

def tokenize(txt):
    toks = okt.morphs(txt)
    toks = [w for w in toks if len(w)>1 and w not in STOPWORDS]
    return toks

def detect_categories(tokens):
    hits=[]
    for tk in tokens:
        for kw, cat in NEG_FLAT.items():
            if kw in tk:
                hits.append(cat)
    return list(set(hits))

def sentiment_score(tokens):
    neg=0
    for tk in tokens:
        for kw in NEG_FLAT.keys():
            if kw in tk:
                neg += 1
    return -neg

print('Text functions ready')

Text functions ready


## 13. Co-occurrence & Token Statistics Utilities

In [13]:
from collections import Counter, defaultdict

def token_stats(df, topn=30):
    all_tokens = [t for toks in df['tokens'] for t in toks]
    return Counter(all_tokens).most_common(topn)

def build_cooccurrence(df, window=4, min_weight=2):
    co = defaultdict(int)
    for toks in df['tokens']:
        L = len(toks)
        for i in range(L):
            for j in range(i+1, min(L, i+window)):
                a,b = sorted([toks[i], toks[j]])
                co[(a,b)] += 1
    return [(a,b,w) for (a,b),w in co.items() if w >= min_weight]

print('Stat utilities ready')

Stat utilities ready


## 14. Visualization Suite (Bars, WordCloud, Category Metrics, Network, Timeline)

In [14]:
def visualize(df):
    if df is None or df.empty:
        print('No data to visualize.')
        return
    # Ensure tokenization
    if 'tokens' not in df:
        df['clean'] = df['content'].map(clean_text)
        df['tokens'] = df['clean'].map(tokenize)
        df['categories'] = df['tokens'].map(detect_categories)
        df['sent_score'] = df['tokens'].map(sentiment_score)
    # Top tokens
    freq = token_stats(df, topn=25)
    if freq:
        fdf = pd.DataFrame(freq, columns=['token','count'])
        plt.figure(figsize=(8,5))
        sns.barplot(data=fdf, x='count', y='token', palette='viridis')
        plt.title('Top Tokens')
        plt.tight_layout(); plt.show()
    # Negative wordcloud
    neg_tokens = []
    for toks in df['tokens']:
        for t in toks:
            if any(kw in t for kw in NEG_FLAT):
                neg_tokens.append(t)
    if neg_tokens:
        wc = WordCloud(width=900,height=450,background_color='white',font_path='C:/Windows/Fonts/malgun.ttf').generate(' '.join(neg_tokens))
        plt.figure(figsize=(10,4)); plt.imshow(wc); plt.axis('off'); plt.title('Negative WordCloud'); plt.show()
    # Category stats
    cat_rows=[]
    for _,r in df.iterrows():
        if isinstance(r.get('categories'), list):
            for c in r['categories']:
                cat_rows.append({'cat':c,'score':r['sent_score']})
    if cat_rows:
        cdf=pd.DataFrame(cat_rows)
        agg = cdf.groupby('cat').agg(freq=('cat','count'), avg_score=('score','mean')).sort_values('freq', ascending=False)
        fig, ax = plt.subplots(1,2, figsize=(12,5))
        sns.barplot(data=agg.reset_index(), x='freq', y='cat', ax=ax[0], palette='magma')
        ax[0].set_title('Category Frequency')
        sns.barplot(data=agg.reset_index(), x='avg_score', y='cat', ax=ax[1], palette='coolwarm')
        ax[1].set_title('Category Avg SentScore')
        plt.tight_layout(); plt.show()
    # Sentiment distribution
    plt.figure(figsize=(6,4))
    sns.histplot(df['sent_score'], bins=20, color='steelblue')
    plt.title('Sentiment Score Distribution')
    plt.tight_layout(); plt.show()
    # Co-occurrence network (negative subset)
    neg_df = df[df['sent_score']<0]
    edges = build_cooccurrence(neg_df)
    if edges:
        G = nx.Graph()
        for a,b,w in edges:
            G.add_edge(a,b,weight=w)
        pos = nx.spring_layout(G, k=0.5, seed=42)
        plt.figure(figsize=(8,6))
        weights=[G[u][v]['weight'] for u,v in G.edges()]
        nx.draw_networkx_nodes(G,pos,node_size=600,node_color='lightcoral')
        nx.draw_networkx_edges(G,pos,width=[1+0.4*w for w in weights],alpha=0.6)
        nx.draw_networkx_labels(G,pos,font_size=10)
        plt.title('Negative Co-occurrence Network')
        plt.axis('off'); plt.show()
    # Timeline
    if 'date_parsed' in df and df['date_parsed'].notna().any():
        tdf = df.dropna(subset=['date_parsed']).groupby(pd.Grouper(key='date_parsed', freq='W')).agg(cnt=('review_id','count'), avg_sent=('sent_score','mean'))
        if not tdf.empty:
            fig, ax1 = plt.subplots(figsize=(10,4))
            ax2 = ax1.twinx()
            ax1.plot(tdf.index, tdf['cnt'], marker='o', color='tab:blue')
            ax2.plot(tdf.index, tdf['avg_sent'], marker='x', color='tab:red')
            ax1.set_ylabel('Review Count'); ax2.set_ylabel('Avg Sent Score')
            ax1.set_title('Weekly Trend')
            fig.tight_layout(); plt.show()
    print('Visualization completed.')

## 15. Pipeline Orchestrator (End-to-End Run)

In [15]:
def enrich_tokens(df):
    df['clean'] = df['content'].map(clean_text)
    df['tokens'] = df['clean'].map(tokenize)
    df['categories'] = df['tokens'].map(detect_categories)
    df['sent_score'] = df['tokens'].map(sentiment_score)
    return df

def run_pipeline(max_reviews=80, max_clicks=20):
    crawl_reviews(max_reviews=max_reviews, max_clicks=max_clicks)
    df = build_processed()
    if df is None:
        return None
    df = enrich_tokens(df)
    df.to_csv(PROC_CSV, index=False, encoding='utf-8-sig')
    df[df['sent_score']<0].to_csv(NEG_CSV, index=False, encoding='utf-8-sig')
    visualize(df)
    return df

# df = run_pipeline(60, 15)

## 16. Quick Unit Tests (Tokenizer & Category Mapping)

In [16]:
def test_sample():
    sample = '배터리가 빨리 닳고 발열이 심하며 키보드 키감이 별로입니다.'
    cl = clean_text(sample)
    toks = tokenize(cl)
    cats = detect_categories(toks)
    score = sentiment_score(toks)
    print('Clean:', cl)
    print('Tokens:', toks)
    print('Categories:', cats)
    print('Sent score:', score)

# test_sample()

## 17. Diagnostic Page Structure Inspector

## 18. Enhanced Debug & Fallback Navigation

## 19. Data Quality & Summary Report Generation

## 20. Large Scale Crawl Execution Cell (Parameterizable)

## 21. Save Artifacts & Export (CSV/PNG) Utility

In [17]:
# 17. Diagnostic Page Structure Inspector
from bs4 import BeautifulSoup as _BS

def inspect_page_structure():
    d = init_driver()
    try:
        d.get(BASE_URL)
        rand_sleep(2,3)
        html = d.page_source
        soup = _BS(html, 'html.parser')
        review_like = soup.find_all(lambda tag: tag.name in ['div','section'] and tag.get('class') and any('review' in ' '.join(tag.get('class')).lower() for _ in [0]))
        print('Candidate review containers:', len(review_like))
        for c in review_like[:3]:
            print('-', c.name, c.get('class')[:3] if c.get('class') else '')
        star_patterns = re.findall(r'[1-5]점|\d\.\d점|별점', html)
        print('Star pattern samples:', list(set(star_patterns))[:10])
    finally:
        d.quit()

# 18. Enhanced Debug & Fallback Navigation

def enhanced_debug(sample=5):
    d = init_driver()
    try:
        d.get(BASE_URL + ('#review' if '#review' not in BASE_URL else ''))
        rand_sleep(2,3)
        sels_try = [SEL['review_item'], '.review', '.user-review', '[data-review]']
        for s in sels_try:
            items = d.find_elements(By.CSS_SELECTOR, s)
            if items:
                print(f'Selector {s} -> {len(items)} items')
                for it in items[:sample]:
                    print(' *', it.text.split('\n')[0][:80])
                break
        more = safe_find(d, By.CSS_SELECTOR, SEL['load_more'])
        print('Load more found' if more else 'Load more NOT found')
    finally:
        d.quit()

# 19. Data Quality & Summary Report Generation

def summary_report(df=None):
    if df is None:
        if not PROC_CSV.exists():
            print('No processed CSV; build first.')
            return
        df = pd.read_csv(PROC_CSV)
    if 'tokens' not in df:
        df = enrich_tokens(df)
    total = len(df)
    dup_ratio = 1 - df['review_id'].nunique()/total if total else 0
    neg_cnt = (df['sent_score']<0).sum()
    cat_counts = {}
    for cats in df['categories']:
        if isinstance(cats, list):
            for c in cats:
                cat_counts[c] = cat_counts.get(c,0)+1
    print(f'Total reviews: {total}')
    print(f'Duplicate ratio: {dup_ratio:.2%}')
    print(f'Negative reviews: {neg_cnt} ({neg_cnt/total:.2% if total else 0})')
    print('Top categories:')
    for k,v in sorted(cat_counts.items(), key=lambda x:x[1], reverse=True)[:10]:
        print(' -', k, v)
    return df

# 20. Large Scale Crawl Execution Cell (Parameterizable)

def large_scale_crawl(target=1000, clicks=120):
    try:
        crawl_reviews(max_reviews=target, max_clicks=clicks)
    except Exception as e:
        print('Large crawl error -> fallback smaller:', e)
        crawl_reviews(max_reviews=min(200,target), max_clicks=min(25,clicks))
    df = build_processed()
    if df is not None:
        df = enrich_tokens(df)
        visualize(df)
    return df

# 21. Save Artifacts & Export (CSV/PNG) Utility

def export_artifacts(df):
    if df is None or df.empty:
        print('No data to export')
        return
    ts = dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    # Save lexicon snapshot
    with open(LEX_SNAPSHOT,'w',encoding='utf-8') as f:
        json.dump(NEGATIVE_KEYWORDS, f, ensure_ascii=False, indent=2)
    # Save parquet optional
    pq = DATA_DIR / f'reviews_{ts}.parquet'
    try:
        df.to_parquet(pq, index=False)
        print('Saved parquet:', pq)
    except Exception as e:
        print('Parquet save failed (pyarrow not installed?)', e)
    # Example figure export (token bar) if already computed in visualize would need adaptation
    print('Artifacts exported.')

print('Sections 17-21 utilities ready')

Sections 17-21 utilities ready


## 22. Extended Analysis & Advanced Visualizations
추가 인사이트: 별점-감정 관계, 리뷰 길이, 카테고리 공존, n-gram, 누적 부정 비율, 레이더 차트, 상관 분석, 텍스트 요약.

In [19]:
import itertools
from math import pi

def ensure_enriched(df):
    if 'tokens' not in df:
        df = enrich_tokens(df)
    return df

def rating_sentiment_plot(df):
    df = ensure_enriched(df)
    if 'rating' in df and df['rating'].notna().any():
        plt.figure(figsize=(6,4))
        sns.boxplot(data=df, x='rating', y='sent_score', palette='coolwarm')
        plt.title('Rating vs Sentiment Score (rule-based)')
        plt.tight_layout(); plt.show()

def review_length_distribution(df):
    df = ensure_enriched(df)
    df['length'] = df['clean'].str.len()
    plt.figure(figsize=(6,4))
    sns.histplot(df['length'], bins=30, color='slateblue')
    plt.title('Review Length Distribution')
    plt.tight_layout(); plt.show()
    if 'rating' in df:
        plt.figure(figsize=(6,4))
        sns.violinplot(data=df, x='rating', y='length', inner='quartile', palette='viridis')
        plt.title('Length by Rating')
        plt.tight_layout(); plt.show()

def category_cooccurrence_heatmap(df, top_k=12):
    df = ensure_enriched(df)
    cat_lists = df['categories'].apply(lambda x: x if isinstance(x,list) else [])
    freq = Counter(itertools.chain.from_iterable(cat_lists))
    top = [c for c,_ in freq.most_common(top_k)]
    if not top:
        print('No categories for heatmap')
        return
    mat = pd.DataFrame(0, index=top, columns=top, dtype=int)
    for cats in cat_lists:
        cats = [c for c in cats if c in top]
        for a,b in itertools.combinations(sorted(set(cats)),2):
            mat.loc[a,b]+=1; mat.loc[b,a]+=1
    plt.figure(figsize=(6,5))
    sns.heatmap(mat, annot=True, fmt='d', cmap='Reds')
    plt.title('Category Co-occurrence (Top)')
    plt.tight_layout(); plt.show()

def top_ngrams(df, n=2, topn=20):
    df = ensure_enriched(df)
    counts = Counter()
    for toks in df['tokens']:
        if len(toks) < n: continue
        for i in range(len(toks)-n+1):
            ng = ' '.join(toks[i:i+n])
            counts[ng]+=1
    return counts.most_common(topn)

def plot_top_ngrams(df):
    for n in (2,3):
        ng = top_ngrams(df, n=n, topn=15)
        if not ng: continue
        ng_df = pd.DataFrame(ng, columns=['ngram','count'])
        plt.figure(figsize=(7,4))
        sns.barplot(data=ng_df, x='count', y='ngram', palette='plasma')
        plt.title(f'Top {n}-grams')
        plt.tight_layout(); plt.show()

def cumulative_negative_trend(df):
    df = ensure_enriched(df)
    if 'date_parsed' not in df or df['date_parsed'].isna().all():
        print('No date info for cumulative trend')
        return
    sdf = df.dropna(subset=['date_parsed']).sort_values('date_parsed')
    sdf['is_negative'] = sdf['sent_score'] < 0
    sdf['cum_cnt'] = range(1,len(sdf)+1)
    sdf['cum_neg'] = sdf['is_negative'].cumsum()
    sdf['cum_neg_ratio'] = sdf['cum_neg']/sdf['cum_cnt']
    plt.figure(figsize=(8,4))
    plt.plot(sdf['date_parsed'], sdf['cum_neg_ratio'], color='crimson')
    plt.title('Cumulative Negative Ratio Over Time')
    plt.ylabel('Cumulative Negative Ratio')
    plt.tight_layout(); plt.show()

def radar_category(df, top_k=6):
    df = ensure_enriched(df)
    cat_rows=[]
    for _,r in df.iterrows():
        if isinstance(r.get('categories'), list):
            for c in r['categories']:
                cat_rows.append(c)
    if not cat_rows:
        print('No categories for radar')
        return
    freq = Counter(cat_rows)
    top = freq.most_common(top_k)
    labels = [c for c,_ in top]
    values = [freq[c] for c in labels]
    values += values[:1]
    angles = [n/float(len(labels))*2*pi for n in range(len(labels))]
    angles += angles[:1]
    plt.figure(figsize=(6,6))
    ax = plt.subplot(111, polar=True)
    plt.xticks(angles[:-1], labels)
    ax.plot(angles, values, linewidth=2, linestyle='solid')
    ax.fill(angles, values, alpha=0.3)
    plt.title('Top Category Radar')
    plt.tight_layout(); plt.show()

def rating_category_heat(df):
    df = ensure_enriched(df)
    if 'rating' not in df or df['rating'].isna().all():
        print('No rating data')
        return
    rows=[]
    for _,r in df.iterrows():
        if isinstance(r.get('categories'), list):
            for c in r['categories']:
                rows.append({'rating':r['rating'], 'cat':c})
    if not rows:
        print('No category rows')
        return
    tdf = pd.DataFrame(rows)
    pivot = tdf.pivot_table(index='cat', columns='rating', aggfunc='size', fill_value=0)
    plt.figure(figsize=(6,5))
    sns.heatmap(pivot, annot=True, fmt='d', cmap='Blues')
    plt.title('Category Frequency by Rating')
    plt.tight_layout(); plt.show()

def sentiment_correlation(df):
    df = ensure_enriched(df)
    if 'rating' not in df or df['rating'].isna().all():
        print('No rating for correlation')
        return
    if 'length' not in df:
        df['length']=df['clean'].str.len()
    corr_df = df[['rating','sent_score','length']].dropna()
    corr = corr_df.corr(numeric_only=True)
    print(corr)
    sns.heatmap(corr, annot=True, cmap='PuOr', center=0)
    plt.title('Correlation Matrix (Rating / Sent / Length)')
    plt.tight_layout(); plt.show()

# Master extended analysis runner

def extended_analysis(df=None):
    if df is None:
        if PROC_CSV.exists():
            df = pd.read_csv(PROC_CSV)
        else:
            print('No processed CSV; run pipeline first.')
            return
    df = ensure_enriched(df)
    print('--- Extended Analysis Start ---')
    rating_sentiment_plot(df)
    review_length_distribution(df)
    category_cooccurrence_heatmap(df)
    plot_top_ngrams(df)
    cumulative_negative_trend(df)
    radar_category(df)
    rating_category_heat(df)
    sentiment_correlation(df)
    print('--- Extended Analysis Done ---')
    return df

# Usage:
# df_ext = extended_analysis()

In [ ]:
# Attempt extended analysis run (safe)
try:
    _df_ext = extended_analysis()
except Exception as e:
    print('Extended analysis skipped:', e)